# Context Strategy Evaluation Runner

Use this notebook in Google Colab to authenticate Hugging Face, run the context strategies, and generate the eval summary table.

Important: `meta-llama/Llama-3.1-8B-Instruct` is gated. Your Hugging Face account must have access before the summarization strategy can load it.

## 1. Runtime Setup

In Colab, choose a GPU runtime first: `Runtime -> Change runtime type -> GPU`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Enter Repo

If the repo is already in Drive, update `REPO_DIR` to that path. Otherwise clone it.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/rissingh23/Context-Opt.git'
REPO_DIR = Path('/content/Context-Opt')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

## 3. Install Dependencies

This installs the project requirements, including `torch`, `transformers`, `bitsandbytes`, and `llmlingua`.

In [ ]:
!python3 -m pip install -r requirements.txt

## 4. Hugging Face Login

Use one of these two options. The account must have access to `meta-llama/Llama-3.1-8B-Instruct`.

In [ ]:
# Option A: interactive login
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Option B: use a Colab secret or environment variable named HF_TOKEN
# Uncomment if you prefer token-based login.
# import os
# from huggingface_hub import login
# login(token=os.environ['HF_TOKEN'])

## 5. Verify Model Access

This checks that the gated Llama config can be reached before we spend time running eval.

In [ ]:
from transformers import AutoConfig

MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
cfg = AutoConfig.from_pretrained(MODEL_NAME)
print('Model access OK:', cfg.model_type)

## 6. Download LongBench Starter Data

This downloads 100 examples per starter task by default. Increase or remove `--limit` later.

In [ ]:
!python3 -m src.data.download_longbench \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --limit 100

## 7. Tiny Smoke Test

Runs one example with all four strategies. This is the first thing to check before a larger eval.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper \
  --strategies full_context retrieval compression summarization \
  --limit 1 \
  --provider mock \
  --model mock_model \
  --summarization-model Qwen/Qwen2.5-7B-Instruct \
  --rows-output outputs/processed/colab_smoke_rows.csv \
  --aggregate-output outputs/processed/colab_smoke_summary.csv \
  --json-output outputs/processed/colab_smoke_rows.jsonl

In [ ]:
import pandas as pd
pd.read_csv('outputs/processed/colab_smoke_summary.csv')

## 8. Full Starter Evaluation

This runs all locally downloaded starter examples. With mock answer model, quality is placeholder-perfect; token, context, and strategy latency are still useful. Replace the answer model provider later when model inference is wired.

In [ ]:
!python3 -m src.eval_framework.run_eval_table \
  --tasks qasper hotpotqa gov_report multi_news passage_count \
  --strategies full_context retrieval compression summarization \
  --limit 500 \
  --provider mock \
  --model mock_model \
  --summarization-model Qwen/Qwen2.5-7B-Instruct \
  --rows-output outputs/processed/colab_eval_rows.csv \
  --aggregate-output outputs/processed/colab_eval_summary.csv \
  --json-output outputs/processed/colab_eval_rows.jsonl

## 9. Render HTML Table

In [ ]:
!python3 tools/render_eval_table.py \
  --input outputs/processed/colab_eval_summary.csv \
  --markdown-output outputs/figures/colab_eval_summary_table.md \
  --html-output outputs/figures/colab_eval_summary_table.html

In [ ]:
from IPython.display import HTML, display
display(HTML(Path('outputs/figures/colab_eval_summary_table.html').read_text()))

## 10. Save Outputs To Drive

Optional: copy result files to Drive.

In [ ]:
DRIVE_OUT = Path('/content/drive/MyDrive/context-opt-outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
!cp -r outputs/processed outputs/figures {DRIVE_OUT}/
print('Copied outputs to', DRIVE_OUT)